# WeatherBench 2 vs. Our Model — Interactive Comparison

Compare the **probabilistic scores** shown on the
[WeatherBench 2 probabilistic-scores page](https://sites.research.google/gr/weatherbench/probabilistic-scores/)
against our own model's scores stored in `parsed_metrics.csv`.

## How it works

The interactive graphic on that website is **not** an HTML table — it is a dynamic plot
that reads pre-computed score files from the public WeatherBench 2 bucket:

```
gs://weatherbench2/benchmark_results/{model}_vs_{baseline}_{resolution}_{year}.nc
```

Each `.nc` file is a small NetCDF holding metrics as a function of `region`, `lead_time` and `level`.
Data variables are named `{metric}.{variable}` (e.g. `crps.geopotential`).

This notebook:

* lists the available files straight from the bucket (no download of the whole dataset),
* **lazily fetches individual score files over HTTPS and caches them in memory for the session only**
  (nothing is written to disk),
* loads our local CSV,
* and lets you overlay the two using dropdown menus.

**Dropdowns for the WeatherBench items:** model, baseline (ground truth), resolution, year,
variable, level, metric, region.

**Dropdowns to filter our model:** variable/feature, metric, model (run).
</VSCode.Cell>


## 1. Import required libraries

We use `xarray` (with the `scipy` NetCDF-3 backend) to read the WeatherBench files,
`pandas` for tabular handling, `plotly` for an interactive figure, and `ipywidgets`
for the dropdown controls. Everything below works inside VS Code notebooks.

If any package is missing, install it into the notebook kernel:

```python
%pip install xarray scipy pandas plotly ipywidgets requests
```


In [ ]:
from __future__ import annotations

import functools
import io
import re
from typing import Optional

import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests
import xarray as xr
from IPython.display import display

pd.set_option("display.max_columns", 50)
print("Libraries imported.")

Libraries imported.


## 2. Data sources and constants

* `WB2_BUCKET` / `WB2_PREFIX` — the public WeatherBench 2 results location.
* `LOCAL_CSV` — path to our own parsed metrics.
* `WB2_VARIABLE_ALIASES` / `LOCAL_FEATURE_MAP` — used to line up variable names and units
  between the two sources.

Adjust `LOCAL_CSV` if your file lives elsewhere.
</VSCode.Cell>


In [ ]:
# --- WeatherBench 2 public results (read over HTTPS, cached in memory only) ---
WB2_BUCKET = "weatherbench2"
WB2_PREFIX = "benchmark_results/"
WB2_JSON_API = f"https://storage.googleapis.com/storage/v1/b/{WB2_BUCKET}/o"
WB2_MEDIA = f"https://storage.googleapis.com/{WB2_BUCKET}"

# --- Our own model scores ---
LOCAL_CSV = "/path/to/parsed_metrics.csv"

# Map our short local feature names -> WeatherBench (variable, level).
# level = None means a single-level / surface variable.
LOCAL_FEATURE_MAP: dict[str, tuple[str, Optional[int]]] = {
    "T2m": ("2m_temperature", None),
    "U10": ("10m_u_component_of_wind", None),
    "V10": ("10m_v_component_of_wind", None),
    "Pmsl": ("mean_sea_level_pressure", None),
    "tp": ("total_precipitation_24hr", None),
    "sst": (
        "sea_surface_temperature",
        None,
    ),  # no direct WB2 equivalent; kept for local-only view
}
# Pressure-level families: local uses e.g. Z_500, T_850, Q_500, U_500, V_500
_LEVEL_FAMILIES = {
    "Z": "geopotential",
    "T": "temperature",
    "Q": "specific_humidity",
    "U": "u_component_of_wind",
    "V": "v_component_of_wind",
}


def local_feature_to_wb2(feature: str) -> tuple[Optional[str], Optional[int]]:
    """Translate a local feature name into a (wb2_variable, level) pair."""
    if feature in LOCAL_FEATURE_MAP:
        return LOCAL_FEATURE_MAP[feature]
    m = re.match(r"^([A-Za-z]+)_(\d+)$", feature)
    if m and m.group(1) in _LEVEL_FAMILIES:
        return _LEVEL_FAMILIES[m.group(1)], int(m.group(2))
    return None, None


print("Constants defined.")
print("Local CSV:", LOCAL_CSV)

## 3. Fetch and cache WeatherBench data (session only)

`list_wb2_files()` reads the bucket listing once and parses each filename into
`model / baseline / resolution / year`.

`_fetch_wb2_dataset()` downloads a **single** score file over HTTPS into an in-memory buffer
and opens it with xarray. Both are wrapped in `functools.lru_cache`, so each file is fetched
**at most once per kernel session** and is never written to disk. Restarting the kernel clears
the cache.
</VSCode.Cell>


In [ ]:
_FNAME_RE = re.compile(
    r"^(?P<model>.+)_vs_(?P<baseline>era5|hres_t0)_(?P<res>\d+x\d+)_(?P<year>\d{4})\.nc$"
)


@functools.lru_cache(maxsize=1)
def list_wb2_files() -> pd.DataFrame:
    """List all benchmark result files in the bucket (one network call, cached)."""
    rows, page_token = [], None
    while True:
        params = {"prefix": WB2_PREFIX, "maxResults": 5000}
        if page_token:
            params["pageToken"] = page_token
        resp = requests.get(WB2_JSON_API, params=params, timeout=60)
        resp.raise_for_status()
        payload = resp.json()
        for obj in payload.get("items", []):
            name = obj["name"]
            fname = name.split("/")[-1]
            m = _FNAME_RE.match(fname)
            if m:
                rows.append({**m.groupdict(), "object": name})
        page_token = payload.get("nextPageToken")
        if not page_token:
            break
    df = pd.DataFrame(rows)
    return df.sort_values(["model", "baseline", "res", "year"]).reset_index(drop=True)


@functools.lru_cache(maxsize=128)
def _fetch_wb2_bytes(object_name: str) -> bytes:
    """Fetch a single object's raw bytes over HTTPS (cached in memory for the session)."""
    url = f"{WB2_MEDIA}/{object_name}"
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    return resp.content


@functools.lru_cache(maxsize=128)
def fetch_wb2_dataset(model: str, baseline: str, res: str, year: str) -> xr.Dataset:
    """Return the xarray Dataset for one WeatherBench score file (cached)."""
    object_name = f"{WB2_PREFIX}{model}_vs_{baseline}_{res}_{year}.nc"
    raw = _fetch_wb2_bytes(object_name)
    # WeatherBench files are classic NetCDF-3 -> the scipy engine handles them without HDF5.
    return xr.open_dataset(io.BytesIO(raw), engine="scipy")


# Load the catalogue now so the dropdowns can be populated.
wb2_catalog = list_wb2_files()
print(f"Found {len(wb2_catalog)} WeatherBench 2 result files.")
print("Models:", ", ".join(sorted(wb2_catalog["model"].unique())))
wb2_catalog.head()

Found 368 WeatherBench 2 result files.
Models: arcana, arches_weather, arches_weather_gen, arches_weather_gen_mean, arches_weather_mx4, aurora, baguan, climatology, ens, ens_mean, ens_single_member, era5_forecast, excarta, fgn_operational, fgn_operational_mean, fgn_operational_single_member, fuxi, gencast, gencast_mean, gencast_operational, gencast_operational_100m_uv, gencast_operational_100m_uv_mean, gencast_operational_100m_uv_single_member, gencast_operational_mean, gencast_operational_single_member, gencast_single_member, graphcast, graphcast_hres_init, hres, keisler, neuralgcm_ens, neuralgcm_ens_mean, neuralgcm_ens_single_member, neuralgcm_hres, pangu, pangu_hres_init, persistence, probabilistic_climatology, stormer_ens_mean, swin, weathermesh4


,model,baseline,res,year,object
0,arcana,era5,1440x721,2022,benchmark_results/arcana_vs_era5_1440x721_2022.nc
1,arcana,era5,240x121,2022,benchmark_results/arcana_vs_era5_240x121_2022.nc
2,arcana,era5,64x32,2022,benchmark_results/arcana_vs_era5_64x32_2022.nc
3,arcana,hres_t0,1440x721,2022,benchmark_results/arcana_vs_hres_t0_1440x721_2...
4,arcana,hres_t0,240x121,2022,benchmark_results/arcana_vs_hres_t0_240x121_20...


## 4. Parse WeatherBench scores into a tidy table

Each file stores data variables named `{metric}.{variable}` over dimensions
`region`, `lead_time` and (for upper-air variables) `level`.

`wb2_to_long()` melts a selected file into a long DataFrame with columns
`metric, variable, level, region, lead_time, value`, which is convenient for filtering and plotting.
We also expose helpers to enumerate the available metrics / variables / levels / regions.


In [ ]:
def _split_varname(name: str) -> tuple[str, str]:
    """'crps.geopotential' -> ('crps', 'geopotential')."""
    metric, _, variable = name.partition(".")
    return metric, variable


@functools.lru_cache(maxsize=128)
def wb2_to_long(model: str, baseline: str, res: str, year: str) -> pd.DataFrame:
    """Melt a WeatherBench file into a long DataFrame (cached per file)."""
    ds = fetch_wb2_dataset(model, baseline, res, year)
    frames = []
    for name, da in ds.data_vars.items():
        metric, variable = _split_varname(name)
        sub = da.to_dataframe(name="value").reset_index()
        sub["metric"] = metric
        sub["variable"] = variable
        if "level" not in sub.columns:
            sub["level"] = np.nan
        frames.append(
            sub[["metric", "variable", "level", "region", "lead_time", "value"]]
        )
    long = pd.concat(frames, ignore_index=True)
    # lead_time may come back as int hours or as a timedelta depending on the file;
    # normalise everything to float hours.
    lt = long["lead_time"]
    if pd.api.types.is_timedelta64_dtype(lt):
        long["lead_time"] = lt.dt.total_seconds() / 3600.0
    else:
        long["lead_time"] = pd.to_numeric(lt, errors="coerce")
    long["source"] = "WeatherBench2"
    long["model"] = model
    long["baseline"] = baseline
    long["res"] = res
    long["year"] = year
    return long


def wb2_metrics(model, baseline, res, year) -> list[str]:
    return sorted(wb2_to_long(model, baseline, res, year)["metric"].unique())


def wb2_variables(model, baseline, res, year) -> list[str]:
    return sorted(wb2_to_long(model, baseline, res, year)["variable"].unique())


def wb2_regions(model, baseline, res, year) -> list[str]:
    df = wb2_to_long(model, baseline, res, year)
    return list(pd.unique(df["region"]))


def wb2_levels(model, baseline, res, year, variable) -> list:
    df = wb2_to_long(model, baseline, res, year)
    lv = df.loc[df["variable"] == variable, "level"].dropna().unique()
    return sorted(int(x) for x in lv)


# Peek at one file to confirm parsing.
_probe = wb2_catalog.iloc[(wb2_catalog["res"] == "64x32").values.argmax()]
_demo = wb2_to_long(_probe["model"], _probe["baseline"], _probe["res"], _probe["year"])
print(
    "Metrics:",
    wb2_metrics(_probe["model"], _probe["baseline"], _probe["res"], _probe["year"]),
)
print(
    "Variables:",
    wb2_variables(_probe["model"], _probe["baseline"], _probe["res"], _probe["year"]),
)
_demo.head()

Metrics: ['acc', 'bias', 'mse', 'prediction_activity', 'rmse', 'seeps', 'vector_rmse']
Variables: ['10m_u_component_of_wind', '10m_v_component_of_wind', '10m_wind', '10m_wind_speed', '2m_temperature', 'geopotential', 'mean_sea_level_pressure', 'specific_humidity', 'temperature', 'total_precipitation_24hr', 'total_precipitation_6hr', 'u_component_of_wind', 'v_component_of_wind', 'wind', 'wind_speed']


,metric,variable,level,region,lead_time,value,source,model,baseline,res,year
0,rmse,total_precipitation_24hr,NaN,global,6.0,0.004500,WeatherBench2,arcana,era5,64x32,2022
1,rmse,total_precipitation_24hr,NaN,tropics,6.0,0.005608,WeatherBench2,arcana,era5,64x32,2022
2,rmse,total_precipitation_24hr,NaN,northern-hemisphere,6.0,0.003654,WeatherBench2,arcana,era5,64x32,2022
3,rmse,total_precipitation_24hr,NaN,southern-hemisphere,6.0,0.003642,WeatherBench2,arcana,era5,64x32,2022
4,rmse,total_precipitation_24hr,NaN,europe,6.0,0.002874,WeatherBench2,arcana,era5,64x32,2022


## 5. Load our local model scores

The CSV has columns like `run, metric, time_h, feature, value, rollout, ...`.
For the comparison we treat:

* `run`   → our **model** dropdown,
* `metric`→ our **metric** dropdown,
* `feature`→ our **variable** dropdown,
* `time_h`→ lead time (hours) on the x-axis,
* `value` → the score.
</VSCode.Cell>


In [ ]:
@functools.lru_cache(maxsize=1)
def load_local_scores(path: str = LOCAL_CSV) -> pd.DataFrame:
    df = pd.read_csv(path)
    # Normalise column names / types we rely on.
    df["metric"] = df["metric"].astype(str)
    df["feature"] = df["feature"].astype(str)
    df["run"] = df["run"].astype(str)
    df["time_h"] = pd.to_numeric(df["time_h"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df


local_df = load_local_scores()
print(f"Loaded {len(local_df):,} rows from local CSV.")
print("Columns:", list(local_df.columns))
print("Runs (models):", sorted(local_df["run"].unique()))
print("Metrics:", sorted(local_df["metric"].unique()))
print("Features:", sorted(local_df["feature"].unique()))
local_df.head()

Loaded 40,320 rows from local CSV.
Columns: ['log_file', 'run', 'job_id', 'nodes', 'gpus', 'mesh_dims', 'rollout', 'metric', 'time_h', 'feature', 'value']
Runs (models): ['final-eval-2022-N128-r15_subs12-gss32_e10s85k', 'final-eval-2022-N32-r15_subs12-gss8_e10s85k', 'final-eval-2022-N384-r15_subs12-gss96_e10s85k', 'final-eval-2022-N768-r15_subs12-gss192_e10s85k']
Metrics: ['crps', 'mse', 'rmse', 'spread_skill_ratio']
Features: ['Pmsl', 'Q_100', 'Q_1000', 'Q_150', 'Q_200', 'Q_250', 'Q_300', 'Q_400', 'Q_50', 'Q_500', 'Q_600', 'Q_700', 'Q_850', 'Q_925', 'T2m', 'T_100', 'T_1000', 'T_150', 'T_200', 'T_250', 'T_300', 'T_400', 'T_50', 'T_500', 'T_600', 'T_700', 'T_850', 'T_925', 'U10', 'U_100', 'U_1000', 'U_150', 'U_200', 'U_250', 'U_300', 'U_400', 'U_50', 'U_500', 'U_600', 'U_700', 'U_850', 'U_925', 'V10', 'V_100', 'V_1000', 'V_150', 'V_200', 'V_250', 'V_300', 'V_400', 'V_50', 'V_500', 'V_600', 'V_700', 'V_850', 'V_925', 'W_100', 'W_1000', 'W_150', 'W_200', 'W_250', 'W_300', 'W_400', 'W_50',

,log_file,run,job_id,nodes,gpus,mesh_dims,rollout,metric,time_h,feature,value
0,final-eval-2022-N128-r15_subs12-gss32_e10s85k_...,final-eval-2022-N128-r15_subs12-gss32_e10s85k,1410113,128,512,"4,32,2,2",15,mse,12,Pmsl,3154.394531
1,final-eval-2022-N128-r15_subs12-gss32_e10s85k_...,final-eval-2022-N128-r15_subs12-gss32_e10s85k,1410113,128,512,"4,32,2,2",15,mse,12,U10,0.719934
2,final-eval-2022-N128-r15_subs12-gss32_e10s85k_...,final-eval-2022-N128-r15_subs12-gss32_e10s85k,1410113,128,512,"4,32,2,2",15,mse,12,V10,0.798360
3,final-eval-2022-N128-r15_subs12-gss32_e10s85k_...,final-eval-2022-N128-r15_subs12-gss32_e10s85k,1410113,128,512,"4,32,2,2",15,mse,12,T2m,0.531290
4,final-eval-2022-N128-r15_subs12-gss32_e10s85k_...,final-eval-2022-N128-r15_subs12-gss32_e10s85k,1410113,128,512,"4,32,2,2",15,mse,12,sst,0.123004


## 6. Prepare aligned series for comparison

Two small helpers extract a single **value-vs-lead-time** series from each source, using
consistent column names (`lead_time`, `value`). The WeatherBench series is picked by
`metric / variable / level / region`; the local series by `metric / feature / run`
(optionally filtered by `rollout`).

> **Units caveat:** WeatherBench variables use SI names (e.g. `geopotential` in m²/s²) while
> our local features may already be in different units (e.g. geopotential *height*). The plot
> shows both on the same axes for shape comparison; enable the *normalize* toggle in the
> dashboard to compare curves regardless of absolute scale.
</VSCode.Cell>


In [ ]:
def get_wb2_series(
    model, baseline, res, year, metric, variable, level, region
) -> pd.DataFrame:
    """Return a lead_time/value series for one WeatherBench selection."""
    df = wb2_to_long(model, baseline, res, year)
    mask = (
        (df["metric"] == metric)
        & (df["variable"] == variable)
        & (df["region"] == region)
    )
    if level is None or (isinstance(level, float) and np.isnan(level)):
        mask &= df["level"].isna()
    else:
        mask &= df["level"] == float(level)
    out = df.loc[mask, ["lead_time", "value"]].dropna().sort_values("lead_time")
    return out.reset_index(drop=True)


def get_local_series(metric, feature, run, rollout=None) -> pd.DataFrame:
    """Return a lead_time/value series for one local selection."""
    df = load_local_scores()
    mask = (df["metric"] == metric) & (df["feature"] == feature) & (df["run"] == run)
    if rollout is not None and "rollout" in df.columns:
        mask &= df["rollout"] == rollout
    out = (
        df.loc[mask, ["time_h", "value"]]
        .rename(columns={"time_h": "lead_time"})
        .dropna()
        .groupby("lead_time", as_index=False)["value"]
        .mean()
        .sort_values("lead_time")
    )
    return out.reset_index(drop=True)


def _normalize(series: pd.Series) -> pd.Series:
    lo, hi = series.min(), series.max()
    if pd.isna(lo) or hi == lo:
        return series * 0.0
    return (series - lo) / (hi - lo)


print("Comparison helpers ready.")

Comparison helpers ready.


## 7. Create the interactive filter widgets

Two groups of dropdowns:

**WeatherBench 2** (one per item on the website): `model`, `baseline` (ground truth),
`resolution`, `year`, `variable`, `level`, `metric`, `region`.
These are *dependent* — changing the model/resolution/year refreshes the metric, variable,
level and region choices from the actual file.

**Our model:** `variable (feature)`, `metric`, `model (run)`, plus an optional `rollout`.

There is also a convenience button to sync the WeatherBench variable/metric to match the local
selection, and a *normalize* toggle.
</VSCode.Cell>


In [ ]:
_DD_LAYOUT = widgets.Layout(width="270px")
_DD_STYLE = {"description_width": "90px"}


def _dd(desc, options, value=None):
    options = list(options)
    if value is None and options:
        value = options[0]
    return widgets.Dropdown(
        description=desc,
        options=options,
        value=value,
        layout=_DD_LAYOUT,
        style=_DD_STYLE,
    )


# ----- WeatherBench 2 dropdowns -----
wb2_model = _dd(
    "Model",
    sorted(wb2_catalog["model"].unique()),
    value="gencast" if "gencast" in set(wb2_catalog["model"]) else None,
)
wb2_baseline = _dd("Ground truth", [])
wb2_res = _dd("Resolution", [])
wb2_year = _dd("Year", [])
wb2_metric = _dd("Metric", [])
wb2_variable = _dd("Variable", [])
wb2_level = _dd("Level", [])
wb2_region = _dd("Region", [])

# ----- Local (our model) dropdowns -----
loc_run = _dd("Model (run)", sorted(local_df["run"].unique()))
loc_metric = _dd(
    "Metric",
    sorted(local_df["metric"].unique()),
    value="crps" if "crps" in set(local_df["metric"]) else None,
)
loc_feature = _dd("Variable", sorted(local_df["feature"].unique()))
_rollouts = (
    sorted(local_df["rollout"].dropna().unique())
    if "rollout" in local_df.columns
    else []
)
loc_rollout = _dd("Rollout", ["(any)"] + [str(r) for r in _rollouts])

# ----- extra controls -----
normalize_tb = widgets.ToggleButton(
    value=False,
    description="Normalize",
    icon="arrows-v",
    tooltip="Min-max normalise both curves to [0,1]",
)
sync_btn = widgets.Button(
    description="Sync WB2 → local",
    icon="link",
    tooltip="Set WeatherBench variable/metric to match the local selection",
)
show_wb2 = widgets.Checkbox(value=True, description="Show WeatherBench")
show_loc = widgets.Checkbox(value=True, description="Show our model")

print("Widgets created.")

Widgets created.


In [ ]:
# Cascading logic: model -> baseline/res/year -> metric/variable/region -> level.
_UPDATING = {"flag": False}  # guard against recursive observer calls


def _set_options(dd: widgets.Dropdown, options, prefer=None):
    options = list(options)
    dd.options = options
    if not options:
        dd.value = None
        return
    if dd.value in options:
        return
    if prefer is not None and prefer in options:
        dd.value = prefer
    else:
        dd.value = options[0]


def _refresh_file_axes(*_):
    """When model changes, refresh baseline / resolution / year choices."""
    sub = wb2_catalog[wb2_catalog["model"] == wb2_model.value]
    _set_options(wb2_baseline, sorted(sub["baseline"].unique()), prefer="era5")
    _set_options(wb2_res, sorted(sub["res"].unique()), prefer="64x32")
    _set_options(wb2_year, sorted(sub["year"].unique()), prefer="2020")
    _refresh_content_axes()


def _refresh_content_axes(*_):
    """When model/baseline/res/year change, refresh metric / variable / region / level."""
    if not (
        wb2_model.value and wb2_baseline.value and wb2_res.value and wb2_year.value
    ):
        return
    # only combos that actually exist as files
    sub = wb2_catalog[
        (wb2_catalog["model"] == wb2_model.value)
        & (wb2_catalog["baseline"] == wb2_baseline.value)
        & (wb2_catalog["res"] == wb2_res.value)
        & (wb2_catalog["year"] == wb2_year.value)
    ]
    if sub.empty:
        _set_options(wb2_metric, [])
        _set_options(wb2_variable, [])
        _set_options(wb2_region, [])
        _set_options(wb2_level, [])
        return
    args = (wb2_model.value, wb2_baseline.value, wb2_res.value, wb2_year.value)
    _set_options(wb2_metric, wb2_metrics(*args), prefer="crps")
    _set_options(wb2_variable, wb2_variables(*args), prefer="geopotential")
    _set_options(wb2_region, wb2_regions(*args), prefer="global")
    _refresh_levels()


def _refresh_levels(*_):
    if not (
        wb2_model.value
        and wb2_baseline.value
        and wb2_res.value
        and wb2_year.value
        and wb2_variable.value
    ):
        return
    levels = wb2_levels(
        wb2_model.value,
        wb2_baseline.value,
        wb2_res.value,
        wb2_year.value,
        wb2_variable.value,
    )
    if levels:
        _set_options(wb2_level, levels, prefer=500)
        wb2_level.disabled = False
    else:
        wb2_level.options = ["(surface)"]
        wb2_level.value = "(surface)"
        wb2_level.disabled = True


wb2_model.observe(_refresh_file_axes, names="value")
for w in (wb2_baseline, wb2_res, wb2_year):
    w.observe(_refresh_content_axes, names="value")
wb2_variable.observe(_refresh_levels, names="value")

# Initialise the cascade once.
_refresh_file_axes()
print("Cascading dropdowns initialised.")
print(
    "WB2:",
    wb2_model.value,
    wb2_baseline.value,
    wb2_res.value,
    wb2_year.value,
    "|",
    wb2_metric.value,
    wb2_variable.value,
    wb2_level.value,
    wb2_region.value,
)

Cascading dropdowns initialised.
WB2: gencast era5 64x32 2020 | crps geopotential 500 global


## 8. Build the comparison plot function

`update_plot()` reads the current widget values, pulls both series, and redraws a
Plotly `FigureWidget` with score on the y-axis and lead time (hours) on the x-axis.
It also prints a short status line and suggests the matching WeatherBench variable/level
for whatever local feature is selected.
</VSCode.Cell>


In [ ]:
# The figure and a status line that the callbacks update in place.
fig = go.FigureWidget()
fig.update_layout(
    template="plotly_white",
    height=460,
    margin=dict(l=60, r=20, t=50, b=50),
    xaxis_title="Lead time (hours)",
    yaxis_title="Score",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
status = widgets.HTML()


def _current_wb2_level():
    lv = wb2_level.value
    if lv in (None, "(surface)"):
        return None
    return float(lv)


def update_plot(*_):
    normalize = normalize_tb.value
    traces, notes = [], []

    # --- WeatherBench series ---
    wb2_ok = False
    if show_wb2.value and all(
        v is not None
        for v in (
            wb2_model.value,
            wb2_baseline.value,
            wb2_res.value,
            wb2_year.value,
            wb2_metric.value,
            wb2_variable.value,
            wb2_region.value,
        )
    ):
        try:
            s = get_wb2_series(
                wb2_model.value,
                wb2_baseline.value,
                wb2_res.value,
                wb2_year.value,
                wb2_metric.value,
                wb2_variable.value,
                _current_wb2_level(),
                wb2_region.value,
            )
            if not s.empty:
                y = _normalize(s["value"]) if normalize else s["value"]
                lvl = (
                    ""
                    if _current_wb2_level() is None
                    else f" @ {int(_current_wb2_level())}"
                )
                name = (
                    f"WB2: {wb2_model.value} · {wb2_metric.value}."
                    f"{wb2_variable.value}{lvl} · {wb2_region.value}"
                )
                traces.append(
                    go.Scatter(
                        x=s["lead_time"],
                        y=y,
                        mode="lines+markers",
                        name=name,
                        line=dict(width=2.5),
                    )
                )
                wb2_ok = True
            else:
                notes.append("no WeatherBench data for this combination")
        except Exception as exc:  # noqa: BLE001
            notes.append(f"WB2 error: {exc}")

    # --- Local series ---
    loc_ok = False
    if show_loc.value and all(
        v is not None for v in (loc_metric.value, loc_feature.value, loc_run.value)
    ):
        rollout = None
        if loc_rollout.value not in (None, "(any)"):
            try:
                rollout = type(local_df["rollout"].dropna().iloc[0])(loc_rollout.value)
            except Exception:  # noqa: BLE001
                rollout = None
        s = get_local_series(
            loc_metric.value, loc_feature.value, loc_run.value, rollout
        )
        if not s.empty:
            y = _normalize(s["value"]) if normalize else s["value"]
            name = f"Ours: {loc_run.value} · {loc_metric.value}.{loc_feature.value}"
            traces.append(
                go.Scatter(
                    x=s["lead_time"],
                    y=y,
                    mode="lines+markers",
                    name=name,
                    line=dict(width=2.5, dash="dash"),
                )
            )
            loc_ok = True
        else:
            notes.append("no local data for this combination")

    # Redraw.
    fig.data = ()
    for t in traces:
        fig.add_trace(t)
    ytitle = "Score (normalised 0-1)" if normalize else "Score"
    fig.update_layout(
        yaxis_title=ytitle,
        title=f"{loc_metric.value or ''} / {wb2_metric.value or ''} vs lead time",
    )

    # Suggest matching WB2 variable for the chosen local feature.
    sug_var, sug_lvl = (
        local_feature_to_wb2(loc_feature.value) if loc_feature.value else (None, None)
    )
    sug = ""
    if sug_var:
        sug = (
            f" &nbsp;|&nbsp; local <b>{loc_feature.value}</b> ≈ WB2 "
            f"<b>{sug_var}</b>" + (f" @ {sug_lvl}" if sug_lvl else " (surface)")
        )
    msg = []
    msg.append("WB2 ✔" if wb2_ok else "WB2 –")
    msg.append("ours ✔" if loc_ok else "ours –")
    note = ("; ".join(notes)) if notes else ""
    status.value = (
        f"<span style='font-family:monospace'>{' | '.join(msg)}</span>"
        f"{sug}" + (f"<br><span style='color:#b45309'>{note}</span>" if note else "")
    )


print("Plot function ready.")

Plot function ready.


## 9. Link the widgets to the plot

Every dropdown/toggle calls `update_plot` on change. The **Sync WB2 → local** button maps
the currently selected local feature to its WeatherBench variable/level and metric (when a
sensible match exists), so you can line the two curves up with a single click.
</VSCode.Cell>


In [ ]:
_ALL_CONTROLS = [
    wb2_model,
    wb2_baseline,
    wb2_res,
    wb2_year,
    wb2_metric,
    wb2_variable,
    wb2_level,
    wb2_region,
    loc_run,
    loc_metric,
    loc_feature,
    loc_rollout,
    normalize_tb,
    show_wb2,
    show_loc,
]

# A shared metric vocabulary so "Sync" can map local -> WB2 metric names.
_METRIC_SYNONYMS = {
    "crps": "crps",
    "rmse": "rmse",  # falls back to mean_rmse below if rmse is absent
    "mse": "mse",
    "spread_skill_ratio": "spread_skill",
    "ssr": "spread_skill",
}


def on_sync(_):
    var, lvl = (
        local_feature_to_wb2(loc_feature.value) if loc_feature.value else (None, None)
    )
    if var and var in list(wb2_variable.options):
        wb2_variable.value = var  # triggers level refresh
        if lvl is not None and lvl in list(wb2_level.options):
            wb2_level.value = lvl
    tgt = _METRIC_SYNONYMS.get(loc_metric.value)
    opts = list(wb2_metric.options)
    if tgt not in opts:
        # graceful fallbacks between deterministic/probabilistic naming
        for alt in (
            tgt,
            "mean_rmse" if tgt == "rmse" else None,
            "rmse" if tgt == "mean_rmse" else None,
        ):
            if alt in opts:
                tgt = alt
                break
    if tgt in opts:
        wb2_metric.value = tgt
    update_plot()


sync_btn.on_click(on_sync)
for c in _ALL_CONTROLS:
    c.observe(update_plot, names="value")

update_plot()  # initial render
print("Widgets linked to plot.")

Widgets linked to plot.


## 10. Interactive dashboard

Run the cell below and use the dropdowns. The **left column** controls the WeatherBench 2 curve
(the items from the website); the **right column** filters our own model. The plot updates live.

> WeatherBench score files are fetched on demand and kept **in memory for this session only** —
> the first time you pick a new model/resolution/year there is a brief download pause, after which
> it is instant. Restart the kernel to clear the cache.
</VSCode.Cell>


In [ ]:
def _titled(title, *rows):
    header = widgets.HTML(f"<b style='font-size:1.02em'>{title}</b>")
    return widgets.VBox(
        [header, *rows],
        layout=widgets.Layout(
            border="1px solid #e5e7eb", padding="8px", margin="0 8px 0 0"
        ),
    )


wb2_panel = _titled(
    "WeatherBench 2 (website items)",
    widgets.HBox([wb2_model, wb2_baseline]),
    widgets.HBox([wb2_res, wb2_year]),
    widgets.HBox([wb2_metric, wb2_region]),
    widgets.HBox([wb2_variable, wb2_level]),
)

local_panel = _titled(
    "Our model (filters)",
    widgets.HBox([loc_run, loc_metric]),
    widgets.HBox([loc_feature, loc_rollout]),
    widgets.HBox([sync_btn]),
)

options_bar = widgets.HBox(
    [show_wb2, show_loc, normalize_tb], layout=widgets.Layout(margin="6px 0")
)

dashboard = widgets.VBox(
    [
        widgets.HBox([wb2_panel, local_panel]),
        options_bar,
        status,
        fig,
    ]
)

display(dashboard)

## 11. (Optional) Current selection as a merged table

Run this to get the two currently selected series merged on `lead_time` — handy for exporting
or eyeballing exact numbers.
</VSCode.Cell>


In [ ]:
def current_comparison_table() -> pd.DataFrame:
    wb2 = get_wb2_series(
        wb2_model.value,
        wb2_baseline.value,
        wb2_res.value,
        wb2_year.value,
        wb2_metric.value,
        wb2_variable.value,
        _current_wb2_level(),
        wb2_region.value,
    ).rename(columns={"value": "weatherbench2"})
    rollout = None if loc_rollout.value in (None, "(any)") else loc_rollout.value
    loc = get_local_series(
        loc_metric.value, loc_feature.value, loc_run.value, rollout
    ).rename(columns={"value": "our_model"})
    merged = pd.merge(wb2, loc, on="lead_time", how="outer").sort_values("lead_time")
    merged.attrs["wb2"] = f"{wb2_model.value}/{wb2_metric.value}.{wb2_variable.value}"
    merged.attrs["local"] = f"{loc_run.value}/{loc_metric.value}.{loc_feature.value}"
    return merged.reset_index(drop=True)


current_comparison_table()

,lead_time,weatherbench2,our_model
0,0,1.404444,NaN
1,6,1.324341,NaN
2,12,1.447046,NaN
3,18,1.481490,NaN
4,24,1.668539,NaN
...,...,...,...
56,336,8.168354,NaN
57,342,8.173530,NaN
58,348,8.206586,NaN
59,354,8.208271,NaN
